In [2]:
import tkinter as tk
from tkinter import messagebox
import random

BG = "#0b0b0b"
GOLD = "#d4af37"
WHITE = "#f5f5f5"
BTN_BLUE = "#1e90ff"
BTN_RED = "#8b0000"

perfumes = {
    "Draker Noir ": 2500,
    "Exclusive": 5200,
    "Signature ": 1800,
    "Royal Marriage": 2200,
    "Aventus Creed": 2000,
    "Schannal-the-Blue": 4500,
    "Gucci Flora":10000,
}

cart = {}
customer_name = ""
customer_id = ""
final_amount = 0
payment_method = ""

def update_output(text):
    output_text.config(state="normal")
    output_text.delete("1.0", tk.END)
    output_text.insert(tk.END, text)
    output_text.config(state="disabled")

def create_account():
    global customer_name, customer_id
    customer_name = name_entry.get()
    if not customer_name:
        messagebox.showerror("Error", "Please enter your name")
        return

    customer_id = random.randint(1000, 9999)
    account_frame.pack_forget()
    shop_frame.pack(pady=10)
    update_output(
        f"Welcome {customer_name}\n"
        f"Customer ID: {customer_id}\n\n"
        f"Select perfumes to continue"
    )

def add_to_cart(item):
    try:
        qty = int(qty_vars[item].get())
        if qty <= 0:
            raise ValueError
    except:
        messagebox.showerror("Error", "Enter valid quantity")
        return

    cart[item] = cart.get(item, 0) + qty
    update_output(f"{item} added\nTotal Quantity: {cart[item]}")

def remove_from_cart(item):
    if item not in cart:
        messagebox.showerror("Error", "Item not in cart")
        return

    try:
        qty = int(qty_vars[item].get())
        if qty <= 0:
            raise ValueError
    except:
        messagebox.showerror("Error", "Enter valid quantity to remove")
        return

    if qty >= cart[item]:
        del cart[item]
        update_output(f"{item} removed from cart")
    else:
        cart[item] -= qty
        update_output(f"{item} remaining quantity: {cart[item]}")

def show_cart():
    if not cart:
        messagebox.showerror("Cart Error", "Cart is empty")
        return
    total = 0
    text = "🛒 CART SUMMARY\n\n"
    for item, qty in cart.items():
        cost = perfumes[item] * qty
        total += cost
        text += f"{item} x {qty} = Rs {cost}\n"
    text += f"\nSubtotal: Rs {total}"
    update_output(text)

def open_billing():
    if not cart:
        messagebox.showerror("Error", "Cart is empty")
        return

    billing_window = tk.Toplevel(root)
    billing_window.title("STARFUMES | Billing")
    billing_window.geometry("600x600")
    billing_window.config(bg=BG)

    tk.Label(billing_window, text="STARFUMES BILL",
             fg=GOLD, bg=BG, font=("Arial", 20, "bold")).pack(pady=10)

    tk.Label(billing_window,
             text=f"Customer: {customer_name}\nCustomer ID: {customer_id}",
             fg=WHITE, bg=BG).pack(pady=5)

    bill_text = tk.Text(billing_window, width=60, height=12,
                        bg="#1a1a1a", fg=WHITE)
    bill_text.pack(pady=10)

    subtotal = 0
    for item, qty in cart.items():
        price = perfumes[item] * qty
        subtotal += price
        bill_text.insert(tk.END, f"{item} x {qty} = Rs {price}\n")
    bill_text.insert(tk.END, f"\nSubtotal: Rs {subtotal}")
    bill_text.config(state="disabled")

    pay_var = tk.StringVar()

    def process_payment():
        global final_amount
        if pay_var.get() == "":
            messagebox.showerror("Error", "Select payment method")
            return

        if pay_var.get() == "COD":
            final_amount = subtotal * 1.05
            show_thank_you(billing_window, final_amount)

        else:
            final_amount = round(subtotal * 1.02, 2)
            ask_card_amount(final_amount, billing_window)

    tk.Radiobutton(billing_window, text="Cash on Delivery (5% Tax)",
                   variable=pay_var, value="COD",
                   bg=BG, fg=WHITE, selectcolor=BG).pack(anchor="w", padx=20)

    tk.Radiobutton(billing_window, text="Card Payment (2% Tax)",
                   variable=pay_var, value="CARD",
                   bg=BG, fg=WHITE, selectcolor=BG).pack(anchor="w", padx=20)

    tk.Button(billing_window, text="Proceed",
              bg=GOLD, command=process_payment).pack(pady=10)

def ask_card_amount(amount, parent):
    win = tk.Toplevel(parent)
    win.title("Card Payment")
    win.geometry("400x300")
    win.config(bg=BG)

    tk.Label(win, text=f"Pay Rs {amount}",
             fg=GOLD, bg=BG, font=("Arial", 16)).pack(pady=10)

    entry = tk.Entry(win)
    entry.pack(pady=10)

    def confirm():
        try:
            if float(entry.get()) == amount:
                show_thank_you(win, amount)
            else:
                messagebox.showerror("Error", "Incorrect amount")
        except:
            messagebox.showerror("Error", "Enter valid amount")

    tk.Button(win, text="Confirm Payment",
              bg=GOLD, command=confirm).pack()

def show_thank_you(parent, amount):
    parent.destroy()
    thank = tk.Toplevel(root)
    thank.title("Order Complete")
    thank.geometry("500x400")
    thank.config(bg=BG)

    tk.Label(thank, text="✨ THANK YOU ✨",
             fg=GOLD, bg=BG, font=("Arial", 22, "bold")).pack(pady=30)

    tk.Label(thank,
             text=f"Your order has been shipped 🚚\n"
                  f"Total Paid: Rs {amount}\n\n"
                  f"Thank you for shopping with STARFUMES",
             fg=WHITE, bg=BG,
             font=("Arial", 14), justify="center").pack()

root = tk.Tk()
root.title("STARFUMES")
root.geometry("800x750")
root.config(bg=BG)

account_frame = tk.Frame(root, bg=BG)
account_frame.pack(pady=30)

tk.Label(account_frame, text="STARFUMES",
         fg=GOLD, bg=BG, font=("Arial", 26, "bold")).pack()

tk.Label(account_frame, text="Enter Your Name",
         fg=WHITE, bg=BG).pack(pady=10)

name_entry = tk.Entry(account_frame, width=30)
name_entry.pack()

tk.Button(account_frame, text="Create Account",
          bg=GOLD, command=create_account).pack(pady=15)

shop_frame = tk.Frame(root, bg=BG)

tk.Label(shop_frame, text="Perfume Collection",
         fg=GOLD, bg=BG, font=("Arial", 18, "bold")).pack(pady=10)

qty_vars = {}

for p, price in perfumes.items():
    row = tk.Frame(shop_frame, bg=BG)
    row.pack(pady=4)

    tk.Label(row, text=f"{p} - Rs {price}",
             fg=WHITE, bg=BG, width=30, anchor="w").pack(side=tk.LEFT)

    qty_vars[p] = tk.Entry(row, width=5)
    qty_vars[p].pack(side=tk.LEFT, padx=5)

    tk.Button(row, text="Add",
              bg=GOLD, command=lambda x=p: add_to_cart(x)).pack(side=tk.LEFT)

    tk.Button(row, text="Delete",
              bg=BTN_RED, fg="white",
              command=lambda x=p: remove_from_cart(x)).pack(side=tk.LEFT)

button_frame = tk.Frame(shop_frame, bg=BG)
button_frame.pack(pady=15)

tk.Button(button_frame, text="View Cart",
          bg=BTN_BLUE, fg="white", width=15,
          command=lambda: show_cart()).pack(side=tk.LEFT, padx=10)

tk.Button(button_frame, text="Proceed to Billing",
          bg=GOLD, fg="black", width=15,
          command=open_billing).pack(side=tk.LEFT, padx=10)

output_frame = tk.Frame(root, bg=BG)
output_frame.pack(pady=10)

tk.Label(output_frame, text="Status",
         fg=GOLD, bg=BG).pack()

output_text = tk.Text(output_frame, height=8, width=70,
                      bg="#1a1a1a", fg=WHITE, state="disabled")
output_text.pack()

root.mainloop()
